# First-Year Debugging Pack — Group Exercises

These snippets intentionally contain **bugs** (syntax, logic, edge cases, scope, mutability, loops, and recursion).

**How to use in groups:**

1. Run each cell *as-is* and read any errors or incorrect outputs.
2. Discuss the likely cause. Write a one-sentence explanation *before* changing code.
3. Fix the bug(s) and re-run.
4. Add at least one `assert` test for each function once fixed.

> Tip: Try edge cases!


## 1) Counting Evens (off-by-one, range bounds)
**Goal:** Count even integers between `start` and `end` inclusive.

**Hints:**
- `range(start, end+1)` covers the end point.
- If `start > end`, consider returning 0 early.


In [2]:
# BUGGY: Count number of even integers in [start, end].
def count_evens(start, end):
    """Returns count of even numbers between start and end inclusive.
    """
    cnt = 0
    # BUG: range end not inclusive; also what if start > end?
    for x in range(start, end):
        if x % 2 == 0:
            cnt += 1
    return cnt

# Tests to try:
# How many evens in numbers from 2 to 6?
print('Expect 3:', count_evens(2, 6))      # evens: 2,4,6 -> 3

# How many evens between 3 and 3, 1 number?
print('Expect 0:', count_evens(3, 3))      # no evens if 3..3

# How many evens from 8 to 8?
print('Expect 1:', count_evens(8, 8))      # only 8 -> 1

# From 9 down to 8?
print('Expect 0:', count_evens(9, 8))      # reversed range should be 0


Expect 3: 3
Expect 0: 0
Expect 1: 1
Expect 0: 0


## 2) Mutability Trap (default list argument)
**Goal:** Collect first `n` unique items. Be careful with default mutable args.

**Hint:** Use `None` default, then create a new list inside.


In [10]:
# BUGGY: default list persists across calls.
def collect_unique(items, n, acc=[]):
    """Returns first n unique items from 'items'.
    """
    for it in items:
        if it not in acc:
            acc.append(it)
        if len(acc) == n:
            break
    return acc

# Tests:
print("Expect ['a', 'b']:", collect_unique(['a', 'a', 'b'], 2))

print("Expect ['x']:", collect_unique(['x', 'x'], 1))  # May include previous run's items!


Expect ['a', 'b']: ['a', 'b']
Expect ['x']: ['a', 'b', 'x']


## 3) Scope & Shadowing (global vs local)
**Goal:** Either use `global` correctly or avoid it by returning values.

UnboundLocalError: cannot access local variable 'total' where it is not associated with a value


In [15]:

total = 0

def add_to_total(nums):
    
    for n in nums:
        total += n

add_to_total([1, 2, 3])


UnboundLocalError: cannot access local variable 'total' where it is not associated with a value

In [17]:
# Local-only version, already works.
# This version avoids global and is usually preferred in real projects.

def add_to_total(total, nums):
    for n in nums:
        total += n
    return total

total = 0
total = add_to_total(total, [1, 2, 3])
# ! Already debugged !


In [21]:
# Or, even better local-only version with try catch.
# This version avoids global and is usually preferred in real projects.

def add_to_total(total, nums):
    for n in nums:
        total += n
    return total

try:
    total = 0
    total = add_to_total(total, [1, 2, 3])
    print('Expect 6:', total)    
except Exception as e:
    print('Raised:', e)

# ! Already debugged ! 

Expect 6: 6


## 4) Hex to Binary (base handling and prefixes)
**Goal:** Convert a hex string like `'FF'` to a binary string without `'0b'`.

**Hints:** Use `int(hx, 16)`; `bin(num)[2:]`.


In [73]:
# BUGGY: Convert hex string to binary string (without prefixes).
def hex_to_bin(hex_str):
    """hx: 'FF' => '11111111'
    """
    
    # Validate that is string
    if not isinstance(hex_str, str):
        raise TypeError("Input must be a string.")
    
     # Validate that the string contains only hex digits
    if not all(c in "0123456789abcedf" for c in hex_str):
        raise ValueError("Invalid hexadecimal string.")
    
    # Remove '0x' prefix if present
    if hex_str.startswith("0x"):
        hex_str = hex_str[2:]
        
    int_str = int(hex_str) # hexadecimal to int needs something.
    binary_str = bin(int_str) # int to binary
        
  

    #bin() includes '0b' prefix; slicing fragile if num is 0
    return binary_str[2:] # 

# Tests:
print('Expect 11111101:', hex_to_bin('fd')) # Capital letter!
print('Expect 101011001111:', hex_to_bin('acf'))
print('Expect 10000:', hex_to_bin('10'))
#print('Expect 1111 or 0b1111:', hex_to_bin('0xf')) # not valid iwth int(hex_str, 16)


ValueError: invalid literal for int() with base 10: 'fd'

### Even "more better" 
This version correctly handles binary, octal, hexadecimal, and decimal, with or without prefixes:


In [75]:


def parse_number(s):
    """
    Safely convert a string to an integer.
    Supports 0b, 0o, 0x prefixes and plain decimals.
    """
    try:
        return int(s, 0)
    except ValueError:
        raise ValueError(f"'{s}' is not a valid numeric literal")


# Try numbr = any of "ox17", "0o77", "0b1011", "34": they are hexadecimal, octal, binary and integer. 
numbr = "34"# requires strings of the numbers.
parsed = parse_number(numbr)

print(f"Parsed number is {parsed}")



Parsed number is 34


## 5) Case-Insensitive Search (method calls)
**Goal:** Return index of first occurrence ignoring case, or `-1`.


In [ ]:
# BUGGY: Find index of first occurrence of 'needle' in 'haystack' ignoring case.
def find_index_ci(haystack, needle):
    h = haystack.lower
    n = needle.lower()
    # Calling .lower incorrectly; also use find()/index()
    pos = h.find(n)
    return pos

# Tests:
print('Expect 0:', find_index_ci('Hello', 'he'))
print('Expect 2:', find_index_ci('abCDef', 'cd'))
print('Expect -1:', find_index_ci('xyz', 'a'))


## 6) Recursion: Factorial (base cases & input validation)
**Goal:** Handle `0!`, `1!`, and reject negatives.


In [ ]:
# BUGGY: Factorial using recursion.
def factorial(n):
    if n == 1:
        return 1
    # misses n == 0, mishandles negatives
    return n * factorial(n - 1)

# Tests:
print('Expect 1:', factorial(1))
print('Expect 120:', factorial(5))
try:
    print('Expect 1:', factorial(0))
except Exception as e:
    print('Raised for 0:', e)
try:
    print('Expect error:', factorial(-3))
except Exception as e:
    print('Raised for -3:', e)


## 7) Queue (FIFO) using list
**Goal:** Ensure `dequeue` removes from the front.


In [ ]:
# BUGGY: Simple queue enqueue/dequeue.
def enqueue(q, item):
    q.append(item)
    return q

def dequeue(q):
    # pop() default removes last (LIFO). Queue needs FIFO.
    return q.pop()

# Tests:
q = []
enqueue(q, 1); enqueue(q, 2); enqueue(q, 3)
print('Expect 1:', dequeue(q))
print('Expect 2:', dequeue(q))
print('Expect 3:', dequeue(q))


## 8) Sorting by Numeric Suffix (key functions)
**Goal:** Sort `['item2','item10','item1']` as `['item1','item2','item10']`.


In [ ]:
# BUGGY: Sort strings numerically by trailing number.
def sort_items(items):
    return sorted(items)  # 'item10' < 'item2' lexicographically

# Tests:
items = ['item2', 'item10', 'item1']
print("Expect ['item1','item2','item10']:", sort_items(items))


## 9) Text Analysis without Files (lines & words)
**Goal:** Count lines and words from a multiline string.

**Hints:** `splitlines()` for lines; `split()` for any whitespace. Handle empty string and trailing newlines.


In [ ]:
# BUGGY: Count lines and words from a multiline string.
def analyze_text(text):
    """Returns (num_lines, num_words).
    """
    # wrong line count when empty or trailing newline
    num_lines = text.count('\n')
    # split on space only; multiple whitespace?
    num_words = len(text.split(' '))
    return num_lines, num_words

# Tests:
print('Expect (1, 2):', analyze_text('hello world'))
print('Expect (2, 3):', analyze_text('a b\nc'))
print('Expect (0, 0):', analyze_text(''))
print('Expect (2, 2):', analyze_text('a\nb\n'))


## 10) While Loop (infinite loop risk)
**Goal:** Find the smallest power of 2 that is `>= n`.


In [ ]:
# BUGGY: Find smallest power of 2 >= n
def smallest_power_of_2(n):
    p = 0
    value = 1
    # if n <= 0? Also loop might not terminate if 'value' never increases properly.
    while value < n:
        value = value  # no change!
        p += 1
    return value, p

# Tests:
print('Expect (1, 0):', smallest_power_of_2(1))
print('Expect (2, 1):', smallest_power_of_2(2))
print('Expect (4, 2):', smallest_power_of_2(3))


## 11) Nested Loops (break levels & indices)
**Goal:** Return `(found, index_of_sublist)` where target appears.


In [ ]:
# BUGGY: Find whether any sublist contains target.
def nested_find(lists, target):
    idx = -1
    for i in range(len(lists)):
        for j in range(len(lists[i])):
            if lists[i][j] == target:
                idx = j  # returns inner index, not sublist index; only breaks inner loop
                break
    return (idx != -1), idx

# Tests:
print('Expect (True, 0):', nested_find([[1,2],[3,4]], 1))
print('Expect (True, 1):', nested_find([[1,2],[3,4]], 3))
print('Expect (False, -1):', nested_find([[1],[2]], 9))


## 12) Recursion + Default Arguments (pattern builder)
**Goal:** Build `['*']`, `['*','**','***']` for `n=1..3`. Avoid shared default list.


In [ ]:
# BUGGY: Build a pattern list like: f(3,'*') -> ['*', '**', '***']
def build_pattern(n, ch='*', acc=[]):
    """Returns a list of strings from 1..n of repeated 'ch'.
    """
    if n == 0:
        return acc
    acc.append(ch * n)
    # Recursion direction inconsistent with desired output
    return build_pattern(n - 1, ch, acc)

# Tests:
print("Expect ['*']:", build_pattern(1))
print("Expect ['*','**','***']:", build_pattern(3))
print('Multiple calls should not share state:', build_pattern(2), build_pattern(2))
